# ᚱ منهج التدريب التراكمي الشامل لنقوش الفايكنج الصخرية (توليد متوازن ونظيف 100%)
### Master 3-Stages Viking Epigraphy Curriculum (YOLOv8)
يقوم هذا الدفتر بتدريب موديل ذكاء اصطناعي خبير في قراءة نقوش الفايكنج الصخرية (Younger Futhark - 17 فئة) وفق **معايير هندسة وتنقيب البيانات (Clean Data Engineering Standards)**:

- ⚖️ **توزيع طبقي متوازن 100% (Stratified Balanced Sampling):** ظهور جميع الحروف الرونية الـ 16 بنفس التكرار تماماً بالتساوي لمنع أي انحياز (Zero Class Imbalance).
- 🧼 **بيانات نظيفة خالية تماماً من الضوضاء العشوائية (Noise-Free Inputs):** إلغاء أي تشويش بكسلي عشوائي لتجنب الـ (Garbage Input) والاعتماد على التدرجات الجيولوجية الصخرية الحقيقية (جرانيت، رملي، أردواز) وتجسيم الحفر ثلاثي الأبعاد الواضح.
- 📐 **أشرطة تأطير متوازية وميلان واقعي بدون تداخل:** مسافات هندسية تمنع تلامس الحروف ومربعات التسمية.
- 💾 **حفظ تلقائي واستئناف آمن في Google Drive:** حفظ تدريجي لكل مرحلة بمجلد `viking_epigraphy_runs`.

---

### مراحل المنهج الثلاثة:
1. **المرحلة الأولى (Stage 1):** تشريح الحروف والسيقان الرونية بكثافة منخفضة وتباين عالي (4-6 حروف/سطر).
2. **المرحلة الثانية (Stage 2):** أشرطة الحفر المؤطرة المتوازية مع الميلان الطبيعي وأنواع الصخور (8-14 حرفاً/سطر).
3. **المرحلة الثالثة (Stage 3):** النقوش الكاملة عالية الكثافة (16-26 حرفاً) مع فواصل الكلمات وتآكل الصخور الطبيعي.


## 1. تثبيت الحزم وربط Google Drive


In [ ]:
# تثبيت مكتبة YOLO والتبعيات
!pip install -q ultralytics pyyaml opencv-python pillow matplotlib seaborn

from google.colab import drive
import os, sys, glob, shutil
from pathlib import Path

# ربط Google Drive
drive.mount('/content/drive')

# تجهيز المجلد الدائم للمشروع
DRIVE_PROJECT = Path('/content/drive/MyDrive/viking_epigraphy_runs')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

print("✅ تم ربط Google Drive وتجهيز مسار العمل بنجاح:")
print(f"👉 {DRIVE_PROJECT}")


In [ ]:
import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device GPU     : {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("⚠️ تنبيه: يرجى تفعيل كرت الشاشة GPU من قائمة: Runtime -> Change runtime type -> T4 GPU")


## 2. محرك التوليد النظيف والمتوازن (Clean & Balanced Engine)
- عينات متوازنة طبقياً (Stratified Round-Robin Sampling).
- أسطح صخرية جيولوجية نظيفة بدون تشويش رقمي (No Artificial Pixel Noise).
- فيزياء تجسيم الإزميل الحجري (Highlight Bevel + Groove Core).


In [ ]:
import random, math, numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
from collections import Counter
import yaml

# 17 فئة أثرية دقيقة (16 حرف إسكندنافي + فاصل الكلمات)
RUNES = [
    {
        "id": 0, "name": "fehu", "latin": "F",  # ᚠ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.45), (0.52, 0.35), (0.75, 0.20)], [(0.35, 0.70), (0.52, 0.60), (0.75, 0.45)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.42), (0.75, 0.18)], [(0.35, 0.68), (0.75, 0.42)]]
        ]
    },
    {
        "id": 1, "name": "uruz", "latin": "U",  # ᚢ (Inverted arch ∩ matching classes.txt)
        "variants": [
            [[(0.30, 0.32), (0.30, 0.90)], [(0.30, 0.32), (0.34, 0.18), (0.50, 0.12), (0.66, 0.18), (0.70, 0.32)], [(0.70, 0.32), (0.70, 0.90)]],
            [[(0.28, 0.30), (0.28, 0.92)], [(0.28, 0.30), (0.50, 0.10), (0.72, 0.30)], [(0.72, 0.30), (0.72, 0.92)]]
        ]
    },
    {
        "id": 2, "name": "thurisaz", "latin": "Th",  # ᚦ (Stave with D-shaped loop)
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.28), (0.55, 0.28), (0.72, 0.38), (0.72, 0.58), (0.55, 0.68), (0.35, 0.68)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.25), (0.75, 0.48), (0.35, 0.72)]]
        ]
    },
    {
        "id": 3, "name": "ansuz", "latin": "A",  # ᚬ (Stave with two crossing downward branches)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.25, 0.28), (0.75, 0.40)], [(0.25, 0.48), (0.75, 0.60)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.26), (0.78, 0.38)], [(0.22, 0.46), (0.78, 0.58)]]
        ]
    },
    {
        "id": 4, "name": "raidho", "latin": "R",  # ᚱ (Stave with loop and leg)
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.14), (0.55, 0.15), (0.70, 0.24), (0.70, 0.38), (0.55, 0.46), (0.35, 0.48)], [(0.35, 0.48), (0.70, 0.90)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.12), (0.72, 0.28), (0.35, 0.48)], [(0.35, 0.48), (0.72, 0.90)]]
        ]
    },
    {
        "id": 5, "name": "kaunan", "latin": "K",  # ᚴ (Stave with upward branch)
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.50), (0.55, 0.38), (0.75, 0.20)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.48), (0.75, 0.18)]]
        ]
    },
    {
        "id": 6, "name": "hagalaz", "latin": "H",  # ᚼ (Stave crossed by asterisk/cross)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.35), (0.78, 0.65)], [(0.22, 0.65), (0.78, 0.35)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.20, 0.38), (0.80, 0.62)], [(0.20, 0.62), (0.80, 0.38)]]
        ]
    },
    {
        "id": 7, "name": "naudiz", "latin": "N",  # ᚾ (Stave with crossing diagonal)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.25, 0.38), (0.75, 0.62)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.40), (0.78, 0.60)]]
        ]
    },
    {
        "id": 8, "name": "isaz", "latin": "I",  # ᛁ (Single vertical stave)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)]],
            [[(0.48, 0.10), (0.48, 0.90)]]
        ]
    },
    {
        "id": 9, "name": "ar_jera", "latin": "A_J",  # ᛅ (Stave crossed by upward diagonal)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.25, 0.62), (0.75, 0.38)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.60), (0.78, 0.40)]]
        ]
    },
    {
        "id": 10, "name": "sowilo", "latin": "S",  # ᛋ (Step shape matching classes.txt exactly: top vertical, middle horizontal, bottom vertical)
        "variants": [
            [[(0.35, 0.12), (0.35, 0.48), (0.65, 0.48), (0.65, 0.88)]],
            [[(0.32, 0.14), (0.32, 0.50), (0.68, 0.50), (0.68, 0.86)]]
        ]
    },
    {
        "id": 11, "name": "tiwaz", "latin": "T",  # ᛏ (Arrowhead pointing up)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.25, 0.35), (0.50, 0.10), (0.75, 0.35)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.20, 0.32), (0.50, 0.10), (0.80, 0.32)]]
        ]
    },
    {
        "id": 12, "name": "berkanan", "latin": "B",  # ᛒ (Stave with double B-lobes)
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.14), (0.55, 0.15), (0.70, 0.24), (0.70, 0.38), (0.55, 0.46), (0.35, 0.48)], [(0.35, 0.48), (0.55, 0.50), (0.70, 0.60), (0.70, 0.74), (0.55, 0.84), (0.35, 0.86)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.12), (0.72, 0.30), (0.35, 0.48)], [(0.35, 0.48), (0.72, 0.68), (0.35, 0.88)]]
        ]
    },
    {
        "id": 13, "name": "mannaz", "latin": "M",  # ᛉ (Stave with two upward branches)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.20, 0.12), (0.50, 0.42)], [(0.80, 0.12), (0.50, 0.42)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.18, 0.10), (0.50, 0.40)], [(0.82, 0.10), (0.50, 0.40)]]
        ]
    },
    {
        "id": 14, "name": "laguz", "latin": "L",  # ᛚ (Stave with downward branch)
        "variants": [
            [[(0.40, 0.10), (0.40, 0.90)], [(0.40, 0.10), (0.70, 0.35)]],
            [[(0.38, 0.10), (0.38, 0.90)], [(0.38, 0.10), (0.72, 0.38)]]
        ]
    },
    {
        "id": 15, "name": "yr", "latin": "Y",  # ᛦ (Stave with two downward branches)
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.20, 0.88), (0.50, 0.58)], [(0.80, 0.88), (0.50, 0.58)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.18, 0.90), (0.50, 0.60)], [(0.82, 0.90), (0.50, 0.60)]]
        ]
    },
]
SEPARATOR_DEF = {
    "id": 16,
    "name": "separator",
    "types": ["colon", "cross", "single_dot"]
}

CLASS_NAMES = [r["name"] for r in RUNES] + [SEPARATOR_DEF["name"]]

class BalancedRuneSampler:
    """يضمن التوزيع الطبقي المتوازن 100% بين جميع الحروف الرونية الـ 16"""
    def __init__(self, runes):
        self.runes = runes
        self.n = len(runes)
        self.pool = []
        self._refill()

    def _refill(self):
        self.pool = list(range(self.n))
        random.shuffle(self.pool)

    def next_rune(self):
        if not self.pool:
            self._refill()
        return self.runes[self.pool.pop()]

def generate_clean_stone_texture(w=640, h=640, rock_type=None):
    if rock_type is None:
        rock_type = random.choice(["granite", "sandstone", "limestone", "slate"])

    if rock_type == "granite":
        base_rgb = np.array([160.0, 155.0, 150.0], dtype=np.float32)
        tint = np.array([random.uniform(-8, 12), random.uniform(-4, 4), random.uniform(-12, -2)])
    elif rock_type == "sandstone":
        base_rgb = np.array([180.0, 160.0, 130.0], dtype=np.float32)
        tint = np.array([random.uniform(5, 20), random.uniform(0, 10), random.uniform(-18, -5)])
    elif rock_type == "limestone":
        base_rgb = np.array([195.0, 192.0, 185.0], dtype=np.float32)
        tint = np.array([random.uniform(-4, 8), random.uniform(-4, 4), random.uniform(-8, 0)])
    else:  # slate
        base_rgb = np.array([90.0, 95.0, 100.0], dtype=np.float32)
        tint = np.array([random.uniform(-8, 4), random.uniform(-4, 4), random.uniform(0, 12)])

    gx = np.linspace(random.uniform(0.94, 1.0), random.uniform(1.0, 1.06), w)[None, :]
    gy = np.linspace(random.uniform(0.94, 1.0), random.uniform(1.0, 1.06), h)[:, None]
    lighting = gy * gx

    c_grid = np.random.uniform(0.88, 1.12, (h // 20 + 1, w // 20 + 1)).astype(np.float32)
    c_map = np.array(
        Image.fromarray((c_grid * 128).astype(np.uint8)).resize((w, h), Image.BICUBIC),
        dtype=np.float32
    ) / 128.0

    f_grid = np.random.uniform(0.94, 1.06, (h // 6 + 1, w // 6 + 1)).astype(np.float32)
    f_map = np.array(
        Image.fromarray((f_grid * 128).astype(np.uint8)).resize((w, h), Image.BILINEAR),
        dtype=np.float32
    ) / 128.0

    color_base = (base_rgb + tint)[None, None, :]
    surface = color_base * (c_map[:, :, None] * 0.70 + f_map[:, :, None] * 0.30) * lighting[:, :, None]
    surface = np.clip(surface, 25, 245).astype(np.uint8)

    img = Image.fromarray(surface)
    img = img.filter(ImageFilter.GaussianBlur(radius=0.5))
    return img

def transform_point(nx, ny, cx, cy, size, angle_deg, shear_x=0.0):
    lx = (nx - 0.5) * size
    ly = (ny - 0.5) * size
    lx += ly * shear_x
    rad = math.radians(angle_deg)
    ca, sa = math.cos(rad), math.sin(rad)
    return (cx + lx * ca - ly * sa, cy + lx * sa + ly * ca)

def draw_clean_chiseled_stroke(draw, p1, p2, groove_c, high_c, line_w, hx, hy):
    draw.line([(p1[0] - hx, p1[1] - hy), (p2[0] - hx, p2[1] - hy)], fill=high_c, width=line_w + 1)
    draw.line([p1, p2], fill=groove_c, width=line_w)
    r = max(1, line_w // 2)
    draw.ellipse([p1[0]-r, p1[1]-r, p1[0]+r, p1[1]+r], fill=groove_c)
    draw.ellipse([p2[0]-r, p2[1]-r, p2[0]+r, p2[1]+r], fill=groove_c)

def render_clean_rune(draw, rune_def, cx, cy, size, angle_deg, line_w, base_dark, shear_x=0.0):
    strokes = random.choice(rune_def["variants"])
    all_pts = []
    hx, hy = 1.6, -1.6
    high_c = (min(255, base_dark[0] + 85), min(255, base_dark[1] + 85), min(255, base_dark[2] + 80))
    groove_c = (max(0, base_dark[0] - 70), max(0, base_dark[1] - 70), max(0, base_dark[2] - 70))

    for stroke in strokes:
        pts = [transform_point(p[0], p[1], cx, cy, size, angle_deg, shear_x) for p in stroke]
        all_pts.extend(pts)
        for i in range(len(pts) - 1):
            draw_clean_chiseled_stroke(draw, pts[i], pts[i+1], groove_c, high_c, line_w, hx, hy)

    xs = [p[0] for p in all_pts]
    ys = [p[1] for p in all_pts]
    pad = line_w + 3
    return (max(0, min(xs) - pad), max(0, min(ys) - pad), min(640, max(xs) + pad), min(640, max(ys) + pad))

def render_clean_separator(draw, cx, cy, size, angle_deg, line_w, base_dark, sep_type=None):
    if sep_type is None:
        sep_type = random.choice(SEPARATOR_DEF["types"])

    groove_c = (max(0, base_dark[0] - 70), max(0, base_dark[1] - 70), max(0, base_dark[2] - 70))
    high_c = (min(255, base_dark[0] + 80), min(255, base_dark[1] + 80), min(255, base_dark[2] + 80))
    dot_r = max(2, line_w)
    all_pts = []

    def T(lx, ly):
        return transform_point(lx / size + 0.5, ly / size + 0.5, cx, cy, size, angle_deg, 0.0)

    if sep_type == "colon":
        p1 = T(0, -size * 0.22)
        p2 = T(0, size * 0.22)
        for p in [p1, p2]:
            all_pts.append(p)
            draw.ellipse([p[0] - dot_r - 1, p[1] - dot_r - 1, p[0] + dot_r + 1, p[1] + dot_r + 1], fill=high_c)
            draw.ellipse([p[0] - dot_r, p[1] - dot_r, p[0] + dot_r, p[1] + dot_r], fill=groove_c)
    elif sep_type == "single_dot":
        all_pts.append((cx, cy))
        draw.ellipse([cx - dot_r - 1, cy - dot_r - 1, cx + dot_r + 1, cy + dot_r + 1], fill=high_c)
        draw.ellipse([cx - dot_r, cy - dot_r, cx + dot_r, cy + dot_r], fill=groove_c)
    else:  # cross
        s = size * 0.22
        pts = [T(0, -s), T(0, s), T(-s, 0), T(s, 0)]
        all_pts.extend(pts)
        draw.line([pts[0], pts[1]], fill=groove_c, width=line_w)
        draw.line([pts[2], pts[3]], fill=groove_c, width=line_w)

    xs = [p[0] for p in all_pts]
    ys = [p[1] for p in all_pts]
    pad = dot_r + line_w + 3
    return (max(0, min(xs) - pad), max(0, min(ys) - pad), min(640, max(xs) + pad), min(640, max(ys) + pad))

def bbox_to_yolo(xmin, ymin, xmax, ymax, img_w=640, img_h=640):
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    bx = (xmin + xmax) / (2.0 * img_w)
    by = (ymin + ymax) / (2.0 * img_h)
    return max(0.001, min(0.999, bx)), max(0.001, min(0.999, by)), max(0.01, min(0.99, bw)), max(0.01, min(0.99, bh))

def build_yaml(dpath):
    yp = os.path.join(dpath, 'data.yaml')
    with open(yp, 'w', encoding='utf-8') as f:
        yaml.dump({
            "path": os.path.abspath(dpath),
            "train": "images/train",
            "val": "images/val",
            "nc": len(CLASS_NAMES),
            "names": CLASS_NAMES
        }, f, default_flow_style=False)
    return yp

print("✅ محرك التوليد النظيف والمتوازن جاهز بنجاح!")


## 3. المرحلة الأولى (Stage 1: تشريح الحروف والسيقان الرونية - توليد متوازن)
- **الهدف:** ترسيخ شكل الحروف الإسكندنافية والزوايا الحادة دون ضوضاء بصرية.
- **التوازن الطبقي:** تكرار متساوٍ تماماً بين جميع الحروف الـ 16.
- **الكثافة:** 4 إلى 6 حروف بالسطر الواحد.
- **التدريب:** 40 حقبة، `lr0=0.008`.
- **الحفظ:** حفظ تلقائي لـ `curriculum_s1_best.pt` في Google Drive.


In [ ]:
d1 = 'dataset_stage1'
os.makedirs(f'{d1}/images/train', exist_ok=True); os.makedirs(f'{d1}/labels/train', exist_ok=True)
os.makedirs(f'{d1}/images/val', exist_ok=True); os.makedirs(f'{d1}/labels/val', exist_ok=True)

sampler_s1 = BalancedRuneSampler(RUNES)
counts_s1 = Counter()

def gen_s1(idx, sampler):
    img = generate_clean_stone_texture(640, 640)
    draw = ImageDraw.Draw(img)
    labels = []
    base_dark = (random.randint(50, 75), random.randint(50, 75), random.randint(50, 75))
    lw = random.randint(3, 4)

    num_rows = random.choice([1, 2])
    row_h = random.randint(72, 88)
    for r in range(num_rows):
        y_c = 190 + r * 220
        rune_s = row_h - 10
        draw.line([(20, y_c - row_h//2), (620, y_c - row_h//2)], fill=(35, 35, 40), width=2)
        draw.line([(20, y_c + row_h//2), (620, y_c + row_h//2)], fill=(35, 35, 40), width=2)

        cx = random.randint(50, 80)
        num_runes = random.randint(4, 6)
        for _ in range(num_runes):
            if cx > 580: break
            rune = sampler.next_rune()
            counts_s1[rune['name']] += 1
            angle = random.uniform(-2.5, 2.5)
            xmin, ymin, xmax, ymax = render_clean_rune(draw, rune, cx, y_c, rune_s, angle, lw, base_dark)
            bx, by, bw, bh = bbox_to_yolo(xmin, ymin, xmax, ymax)
            labels.append(f"{rune['id']} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}")
            cx += random.randint(int(rune_s * 0.88), int(rune_s * 1.05))

    return img, labels

print("🚀 جاري توليد بيانات المرحلة الأولى بتوزيع طبقي متوازن 100%...")
for i in range(800):
    im, lbls = gen_s1(i, sampler_s1)
    im.save(f'{d1}/images/train/s1_{i:04d}.jpg', quality=95)
    with open(f'{d1}/labels/train/s1_{i:04d}.txt', 'w') as f: f.write('\n'.join(lbls)+'\n')

for i in range(120):
    im, lbls = gen_s1(1000 + i, sampler_s1)
    im.save(f'{d1}/images/val/s1_val_{i:04d}.jpg', quality=95)
    with open(f'{d1}/labels/val/s1_val_{i:04d}.txt', 'w') as f: f.write('\n'.join(lbls)+'\n')

y1 = build_yaml(d1)

print("📊 تقرير توازن الفئات في بيانات المرحلة الأولى:")
for name, cnt in sorted(counts_s1.items()):
    print(f"   {name:12s}: {cnt} عينة")

from ultralytics import YOLO

m1 = YOLO('yolov8m.pt')
m1.train(
    data=y1,
    epochs=40,
    imgsz=640,
    batch=16,
    lr0=0.008,
    degrees=8.0,
    project=str(DRIVE_PROJECT),
    name='curriculum_s1',
    exist_ok=True,
    save=True
)

s1_best = DRIVE_PROJECT / 'curriculum_s1' / 'weights' / 'best.pt'
s1_backup = DRIVE_PROJECT / 'viking_curriculum_s1_best.pt'
if s1_best.exists():
    shutil.copy2(s1_best, s1_backup)
    print(f"🎉 اكتمل تدريب المرحلة 1 وحُفظ بنجاح في Google Drive:\n👉 {s1_backup}")


## 4. المرحلة الثانية (Stage 2: الأسطر الحجرية المؤطرة والميلان الطبيعي)
- **الهدف:** تدريب الموديل على قراءة الأسطر المؤطرة بأشرطة حفر مائلة بأنماط جيولوجية صخرية نظيفة.
- **التوازن الطبقي:** تكرار متساوٍ بين الحروف مع فواصل كلمات موزونة بدقة.
- **الكثافة:** 8 إلى 14 حرفاً بالسطر مع فواصل كلمات.
- **التدريب:** 50 حقبة، `lr0=0.004`، مع انطلاق من أوزان المرحلة 1 (`s1_best`).
- **الحفظ:** حفظ تلقائي لـ `viking_curriculum_s2_best.pt`.


In [ ]:
d2 = 'dataset_stage2'
os.makedirs(f'{d2}/images/train', exist_ok=True); os.makedirs(f'{d2}/labels/train', exist_ok=True)
os.makedirs(f'{d2}/images/val', exist_ok=True); os.makedirs(f'{d2}/labels/val', exist_ok=True)

sampler_s2 = BalancedRuneSampler(RUNES)
counts_s2 = Counter()

def gen_s2(idx, sampler):
    img = generate_clean_stone_texture(640, 640)
    draw = ImageDraw.Draw(img)
    labels = []
    base_dark = (random.randint(45, 75), random.randint(45, 75), random.randint(45, 75))
    lw = random.randint(3, 4)

    num_rows = random.choice([1, 2])
    row_h = random.randint(65, 82)
    band_angle = random.uniform(-7.5, 7.5)
    rad = math.radians(band_angle)
    cos_a, sin_a, tan_a = math.cos(rad), math.sin(rad), math.tan(rad)
    cx_mid = 320.0

    for r in range(num_rows):
        y_center = 185 + r * (row_h + 85)
        rune_size = row_h - 10
        half_h = row_h / 2.0

        y_l = y_center + (15.0 - cx_mid) * tan_a
        y_r = y_center + (625.0 - cx_mid) * tan_a
        draw.line([(15, y_l - half_h/cos_a), (625, y_r - half_h/cos_a)], fill=(35, 35, 40), width=2)
        draw.line([(15, y_l + half_h/cos_a), (625, y_r + half_h/cos_a)], fill=(35, 35, 40), width=2)

        t_curr = 45.0
        max_t = 550.0 / max(0.3, abs(cos_a))
        runes_since_sep = 0

        while t_curr < max_t:
            rcx = cx_mid + (t_curr - 320.0) * cos_a
            rcy = y_center + (t_curr - 320.0) * sin_a
            if rcx < 35 or rcx > 605 or rcy < 35 or rcy > 605:
                t_curr += rune_size * 0.90
                continue

            r_angle = band_angle + random.uniform(-4.5, 4.5)
            if runes_since_sep >= random.randint(3, 5) and random.random() < 0.75:
                xmin, ymin, xmax, ymax = render_clean_separator(draw, rcx, rcy, rune_size, r_angle, lw, base_dark)
                bx, by, bw, bh = bbox_to_yolo(xmin, ymin, xmax, ymax)
                labels.append(f"16 {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}")
                counts_s2['separator'] += 1
                t_curr += random.randint(int(rune_size * 0.65), int(rune_size * 0.76))
                runes_since_sep = 0
            else:
                rune = sampler.next_rune()
                counts_s2[rune['name']] += 1
                xmin, ymin, xmax, ymax = render_clean_rune(draw, rune, rcx, rcy, rune_size, r_angle, lw, base_dark)
                bx, by, bw, bh = bbox_to_yolo(xmin, ymin, xmax, ymax)
                labels.append(f"{rune['id']} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}")
                t_curr += random.randint(int(rune_size * 0.86), int(rune_size * 1.04))
                runes_since_sep += 1

    return img, labels

print("🚀 جاري توليد بيانات المرحلة الثانية بتوزيع طبقي متوازن...")
for i in range(800):
    im, lbls = gen_s2(i, sampler_s2)
    im.save(f'{d2}/images/train/s2_{i:04d}.jpg', quality=95)
    with open(f'{d2}/labels/train/s2_{i:04d}.txt', 'w') as f: f.write('\n'.join(lbls)+'\n')

for i in range(120):
    im, lbls = gen_s2(1000 + i, sampler_s2)
    im.save(f'{d2}/images/val/s2_val_{i:04d}.jpg', quality=95)
    with open(f'{d2}/labels/val/s2_val_{i:04d}.txt', 'w') as f: f.write('\n'.join(lbls)+'\n')

y2 = build_yaml(d2)

print("📊 تقرير توازن الفئات في بيانات المرحلة الثانية:")
for name, cnt in sorted(counts_s2.items()):
    print(f"   {name:12s}: {cnt} عينة")

s1_candidates = [
    DRIVE_PROJECT / 'curriculum_s1' / 'weights' / 'best.pt',
    DRIVE_PROJECT / 'viking_curriculum_s1_best.pt',
    'yolov8m.pt'
]
s1_path = next((str(p) for p in s1_candidates if Path(p).exists()), 'yolov8m.pt')
print(f"✅ بدء تدريب المرحلة 2 انطلاقاً من: {s1_path}")

m2 = YOLO(s1_path)
m2.train(
    data=y2,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.004,
    mosaic=0.5,
    project=str(DRIVE_PROJECT),
    name='curriculum_s2',
    exist_ok=True,
    save=True
)

s2_best = DRIVE_PROJECT / 'curriculum_s2' / 'weights' / 'best.pt'
s2_backup = DRIVE_PROJECT / 'viking_curriculum_s2_best.pt'
if s2_best.exists():
    shutil.copy2(s2_best, s2_backup)
    print(f"🎉 اكتمل تدريب المرحلة 2 وحُفظ بنجاح في Google Drive:\n👉 {s2_backup}")


## 5. المرحلة الثالثة (Stage 3: الكثافة العالية والنقوش الأثرية الكاملة)
- **الهدف:** تدريب الموديل على المشاهد الأثرية الكاملة (ميلان طبيعي حتى 11 درجة، سطور متعددة وفواصل كلمات).
- **التوازن الطبقي:** تكرار متساوٍ بين جميع الفئات.
- **الكثافة:** 16 إلى 26 حرفاً بالصخرة.
- **التدريب:** 60 حقبة، `lr0=0.002` (صقل فائق الدقة للأوزان).
- **الحفظ:** إنتاج الموديل الخبير النهائي `viking_master_epigraphy_best.pt`.


In [ ]:
d3 = 'dataset_stage3'
os.makedirs(f'{d3}/images/train', exist_ok=True); os.makedirs(f'{d3}/labels/train', exist_ok=True)
os.makedirs(f'{d3}/images/val', exist_ok=True); os.makedirs(f'{d3}/labels/val', exist_ok=True)

sampler_s3 = BalancedRuneSampler(RUNES)
counts_s3 = Counter()

def apply_natural_patina(img):
    if random.random() < 0.60:
        w, h = img.size
        overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
        o_draw = ImageDraw.Draw(overlay)
        lx, ly, lr = random.randint(50, w-50), random.randint(50, h-50), random.randint(40, 100)
        col = random.choice([(70, 95, 65, 40), (135, 125, 90, 45)])
        o_draw.ellipse([lx-lr, ly-lr, lx+lr, ly+lr], fill=col)
        overlay = overlay.filter(ImageFilter.GaussianBlur(radius=random.uniform(15, 25)))
        img = Image.alpha_composite(img.convert("RGBA"), overlay).convert("RGB")
    return img

def gen_s3(idx, sampler):
    img = generate_clean_stone_texture(640, 640)
    draw = ImageDraw.Draw(img)
    labels = []
    base_dark = (random.randint(45, 75), random.randint(45, 75), random.randint(45, 75))
    lw = random.randint(3, 4)

    num_rows = random.randint(2, 3)
    row_h = random.randint(60, 75)
    band_angle = random.uniform(-11.0, 11.0)
    rad = math.radians(band_angle)
    cos_a, sin_a, tan_a = math.cos(rad), math.sin(rad), math.tan(rad)
    cx_mid = 320.0
    start_y = random.randint(90, 115)

    for r in range(num_rows):
        y_center = start_y + r * (row_h + random.randint(30, 42))
        rune_size = row_h - 10
        half_h = row_h / 2.0

        y_l = y_center + (15.0 - cx_mid) * tan_a
        y_r = y_center + (625.0 - cx_mid) * tan_a
        draw.line([(15, y_l - half_h/cos_a), (625, y_r - half_h/cos_a)], fill=(35, 35, 40), width=2)
        draw.line([(15, y_l + half_h/cos_a), (625, y_r + half_h/cos_a)], fill=(35, 35, 40), width=2)

        t_curr = 45.0
        max_t = 555.0 / max(0.3, abs(cos_a))
        runes_since_sep = 0

        while t_curr < max_t:
            rcx = cx_mid + (t_curr - 320.0) * cos_a
            rcy = y_center + (t_curr - 320.0) * sin_a
            if rcx < 35 or rcx > 605 or rcy < 35 or rcy > 605:
                t_curr += rune_size * 0.90
                continue

            r_angle = band_angle + random.uniform(-4.5, 4.5)
            if runes_since_sep >= random.randint(3, 5) and random.random() < 0.85:
                xmin, ymin, xmax, ymax = render_clean_separator(draw, rcx, rcy, rune_size, r_angle, lw, base_dark)
                bx, by, bw, bh = bbox_to_yolo(xmin, ymin, xmax, ymax)
                labels.append(f"16 {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}")
                counts_s3['separator'] += 1
                t_curr += random.randint(int(rune_size * 0.65), int(rune_size * 0.76))
                runes_since_sep = 0
            else:
                rune = sampler.next_rune()
                counts_s3[rune['name']] += 1
                xmin, ymin, xmax, ymax = render_clean_rune(draw, rune, rcx, rcy, rune_size, r_angle, lw, base_dark)
                bx, by, bw, bh = bbox_to_yolo(xmin, ymin, xmax, ymax)
                labels.append(f"{rune['id']} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}")
                t_curr += random.randint(int(rune_size * 0.86), int(rune_size * 1.04))
                runes_since_sep += 1

    return apply_natural_patina(img), labels

print("🚀 جاري توليد بيانات المرحلة الثالثة بتوزيع طبقي متوازن...")
for i in range(1000):
    im, lbls = gen_s3(i, sampler_s3)
    im.save(f'{d3}/images/train/s3_{i:04d}.jpg', quality=95)
    with open(f'{d3}/labels/train/s3_{i:04d}.txt', 'w') as f: f.write('\n'.join(lbls)+'\n')

for i in range(150):
    im, lbls = gen_s3(1000 + i, sampler_s3)
    im.save(f'{d3}/images/val/s3_val_{i:04d}.jpg', quality=95)
    with open(f'{d3}/labels/val/s3_val_{i:04d}.txt', 'w') as f: f.write('\n'.join(lbls)+'\n')

y3 = build_yaml(d3)

print("📊 تقرير توازن الفئات في بيانات المرحلة الثالثة:")
for name, cnt in sorted(counts_s3.items()):
    print(f"   {name:12s}: {cnt} عينة")

s2_candidates = [
    DRIVE_PROJECT / 'curriculum_s2' / 'weights' / 'best.pt',
    DRIVE_PROJECT / 'viking_curriculum_s2_best.pt',
    DRIVE_PROJECT / 'viking_curriculum_s1_best.pt',
    'yolov8m.pt'
]
s2_path = next((str(p) for p in s2_candidates if Path(p).exists()), 'yolov8m.pt')
print(f"🚀 بدء تدريب المرحلة 3 انطلاقاً من: {s2_path}")

m3 = YOLO(s2_path)
m3.train(
    data=y3,
    epochs=60,
    imgsz=640,
    batch=16,
    lr0=0.002,
    mosaic=0.5,
    project=str(DRIVE_PROJECT),
    name='master_viking_stage3',
    exist_ok=True,
    save=True
)

master_best = DRIVE_PROJECT / 'master_viking_stage3' / 'weights' / 'best.pt'
master_backup = DRIVE_PROJECT / 'viking_master_epigraphy_best.pt'
if master_best.exists():
    shutil.copy2(master_best, master_backup)
    print("=" * 75)
    print(f"🎉🎉🎉 تم الانتهاء بنجاح وحفظ الموديل الخبير النهائي في Google Drive:")
    print(f"👉 {master_best}")
    print(f"📁 نسخة رئيسية جاهزة للاستخدام: {master_backup}")
    print("=" * 75)


## 6. تنزيل الموديل الخبير النهائي (اختياري)
يمكنك تشغيل الخلية التالية لتنزيل أوزان الموديل الخبير النهائي `viking_master_epigraphy_best.pt` إلى جهازك مباشرة بنقرة واحدة.


In [ ]:
from google.colab import files

master_file = DRIVE_PROJECT / 'viking_master_epigraphy_best.pt'
if master_file.exists():
    print(f"📥 جاري بدء تنزيل {master_file.name} لجهازك...")
    files.download(str(master_file))
else:
    print("⚠️ لم يتم العثور على الملف، تأكد من اكتمال تشغيل المرحلة 3 أولاً.")
